# 00 — Exploratory Data Analysis: DREAMER.mat

**Goal:** understand the raw DREAMER dataset *before* any modelling — what is in the
`.mat` file, how the participants rated the film clips, what the EEG looks like, and
which data-quality / class-balance issues will matter downstream.

DREAMER (Katsigiannis & Ramzan, 2018; *University of the West of Scotland, 2017*):

| Property | Value |
|---|---|
| Participants | 23 |
| Film clips (trials) per participant | 18 |
| EEG | 14 channels (Emotiv EPOC), 128 Hz |
| ECG | 2 channels, 256 Hz |
| Per trial | a *baseline* (neutral clip, ~61 s) **and** a *stimulus* (emotional clip, last ~60 s used) |
| Self-assessment (SAM) | Valence, Arousal, Dominance, each **1–5** |

The file is a **MATLAB v5 (legacy) container**, so `scipy.io.loadmat` reads it directly
(no `mat73` needed).

This notebook only *reads* `data/raw/DREAMER.mat`. It writes figures to
`results/figures/eda/` and changes nothing else.

## 1. Setup & load

In [ ]:
import os, sys, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.io import loadmat
from scipy.signal import welch, butter, sosfiltfilt

warnings.filterwarnings("ignore", category=RuntimeWarning)
sns.set_theme(context="notebook", style="whitegrid")
plt.rcParams["figure.dpi"] = 110

RAW_PATH = "../data/raw/DREAMER.mat"
FIG_DIR  = "../results/figures/eda"
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(name):
    plt.savefig(os.path.join(FIG_DIR, name), bbox_inches="tight", dpi=130)

EEG_FS, ECG_FS = 128, 256
N_SUB, N_TRIAL = 23, 18
print("reading", RAW_PATH, f"({os.path.getsize(RAW_PATH)/1e6:.0f} MB) ...")
raw = loadmat(RAW_PATH, struct_as_record=False, squeeze_me=True)
D = raw["DREAMER"]
print("DREAMER fields :", D._fieldnames)
CH = list(D.EEG_Electrodes)
print("EEG electrodes :", CH)
print("EEG fs =", D.EEG_SamplingRate, "| ECG fs =", D.ECG_SamplingRate,
      "| subjects =", D.noOfSubjects, "| videos =", D.noOfVideoSequences)
print("version =", D.Version, "| provider =", D.Provider)
data = np.atleast_1d(D.Data)

### 1.1 Flatten into tidy tables

Two long-format frames make everything downstream a one-liner:

* `ratings` — one row per (subject, video): age, gender, V/A/D scores, EEG lengths.
* per-signal arrays are pulled on demand from `data[...]` to keep memory sane.

In [ ]:
rows = []
for si, s in enumerate(data, start=1):
    v = np.asarray(s.ScoreValence,   float).ravel()
    a = np.asarray(s.ScoreArousal,   float).ravel()
    d = np.asarray(s.ScoreDominance, float).ravel()
    eeg_stim = np.atleast_1d(s.EEG.stimuli)
    eeg_base = np.atleast_1d(s.EEG.baseline)
    for ti in range(N_TRIAL):
        st = np.asarray(eeg_stim[ti], float)
        bl = np.asarray(eeg_base[ti], float)
        rows.append(dict(
            subject=si, video=ti + 1,
            age=int(s.Age), gender=str(s.Gender),
            valence=v[ti], arousal=a[ti], dominance=d[ti],
            stim_samples=st.shape[0], base_samples=bl.shape[0],
            stim_sec=st.shape[0] / EEG_FS, base_sec=bl.shape[0] / EEG_FS,
        ))
ratings = pd.DataFrame(rows)
print(ratings.shape, "rows  (23 subjects x 18 videos = 414)")
ratings.head(6)

In [ ]:
# subject-level participant table
subj = (ratings.groupby("subject")
        .agg(age=("age", "first"), gender=("gender", "first"),
             valence=("valence", "mean"), arousal=("arousal", "mean"),
             dominance=("dominance", "mean"),
             total_stim_min=("stim_sec", lambda x: x.sum() / 60))
        .reset_index())
print("Age  : mean {:.1f}  sd {:.1f}  range {}-{}".format(
    subj.age.mean(), subj.age.std(), subj.age.min(), subj.age.max()))
print("Gender:", subj.gender.value_counts().to_dict())
print("Total EEG stimulus recording: {:.0f} min across all subjects".format(subj.total_stim_min.sum()))
subj

## 2. Participants

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(subj.age, bins=np.arange(20, 36, 2), edgecolor="white")
ax[0].set(title="Age distribution", xlabel="age", ylabel="# participants")
ax[0].axvline(subj.age.mean(), color="crimson", ls="--", label=f"mean {subj.age.mean():.1f}")
ax[0].legend()

gc = subj.gender.value_counts()
ax[1].bar(gc.index, gc.values, color=["#4C72B0", "#DD8452"])
ax[1].set(title="Gender balance", ylabel="# participants")
for i, val in enumerate(gc.values):
    ax[1].text(i, val + 0.1, str(val), ha="center")

sns.boxplot(data=ratings, x="gender", y="valence", ax=ax[2])
sns.stripplot(data=ratings, x="gender", y="valence", ax=ax[2], color="k", alpha=.25, size=3)
ax[2].set(title="Valence rating by gender")
plt.suptitle("DREAMER — participant demographics", y=1.03, fontsize=13)
savefig("01_participants.png"); plt.show()

## 3. Affective ratings (Valence / Arousal / Dominance)

414 SAM ratings on a 1–5 scale. These become the **prediction targets**, so their
distribution and balance drive everything.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for i, dim in enumerate(["valence", "arousal", "dominance"]):
    vc = ratings[dim].value_counts().reindex(range(1, 6), fill_value=0)
    ax[i].bar(vc.index, vc.values, edgecolor="white")
    ax[i].set(title=f"{dim.capitalize()}  (mean {ratings[dim].mean():.2f}, sd {ratings[dim].std():.2f})",
              xlabel="SAM score", ylabel="# trials", xticks=range(1, 6))
    for x, y in zip(vc.index, vc.values):
        ax[i].text(x, y + 2, str(y), ha="center", fontsize=9)
plt.suptitle("Rating distributions across all 414 trials", y=1.03, fontsize=13)
savefig("02_rating_distributions.png"); plt.show()

### 3.1 Valence–Arousal space (the affective circumplex)

Where do the film-clip responses land? Jitter added so overlapping integer ratings
are visible; marker colour = dominance.

In [ ]:
rng = np.random.default_rng(0)
jx = ratings.valence + rng.uniform(-.18, .18, len(ratings))
jy = ratings.arousal + rng.uniform(-.18, .18, len(ratings))

fig, ax = plt.subplots(1, 2, figsize=(13, 5.2))
sc = ax[0].scatter(jx, jy, c=ratings.dominance, cmap="viridis", s=28, alpha=.75, edgecolor="none")
ax[0].axvline(3, color="grey", lw=1); ax[0].axhline(3, color="grey", lw=1)
ax[0].set(xlabel="Valence", ylabel="Arousal", title="Valence–Arousal (colour = Dominance)",
          xlim=(.5, 5.5), ylim=(.5, 5.5))
for tx, ty, lbl in [(4.5, 4.5, "excited /\nhappy"), (1.5, 4.5, "angry /\nafraid"),
                    (1.5, 1.5, "sad /\nbored"), (4.5, 1.5, "calm /\ncontent")]:
    ax[0].text(tx, ty, lbl, ha="center", va="center", fontsize=9, color="crimson", alpha=.8)
fig.colorbar(sc, ax=ax[0], label="Dominance")

hb = ax[1].hexbin(ratings.valence, ratings.arousal, gridsize=12, cmap="magma", mincnt=1)
ax[1].set(xlabel="Valence", ylabel="Arousal", title="Density (hexbin)")
fig.colorbar(hb, ax=ax[1], label="# trials")
savefig("03_valence_arousal_space.png"); plt.show()

### 3.2 How the three dimensions relate

In [ ]:
corr = ratings[["valence", "arousal", "dominance"]].corr()
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, vmin=-1, vmax=1,
            square=True, ax=ax[0], cbar_kws=dict(shrink=.8))
ax[0].set_title("Pearson correlation of SAM dimensions")

sns.regplot(data=ratings, x="arousal", y="dominance", ax=ax[1],
            x_jitter=.15, y_jitter=.15, scatter_kws=dict(alpha=.3, s=18),
            line_kws=dict(color="crimson"))
ax[1].set_title(f"Arousal vs Dominance  (r = {corr.loc['arousal','dominance']:.2f})  — the one strong pair")
savefig("04_rating_correlations.png"); plt.show()
print(corr.round(3))

### 3.3 Per-subject rating style

Some participants use the whole 1–5 range; others cluster around the middle. This
*rater bias* is exactly why several benchmark models normalise per subject and why
LOSO (leave-one-subject-out) is hard.

In [ ]:
order = ratings.groupby("subject").valence.median().sort_values().index
fig, ax = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
for i, dim in enumerate(["valence", "arousal", "dominance"]):
    sns.boxplot(data=ratings, x="subject", y=dim, order=order, ax=ax[i], color="#88CCEE")
    sns.stripplot(data=ratings, x="subject", y=dim, order=order, ax=ax[i],
                  color="k", alpha=.35, size=3)
    ax[i].set(ylabel=dim, ylim=(.5, 5.5))
    ax[i].axhline(3, color="crimson", ls="--", lw=1)
ax[-1].set_xlabel("subject (sorted by median valence)")
plt.suptitle("Rating distribution per participant", y=1.0, fontsize=13)
savefig("05_per_subject_ratings.png"); plt.show()

### 3.4 Per-video ratings — do the clips work?

If the film clips elicit consistent emotions, each video should have a stable mean
rating across the 23 subjects.

In [ ]:
pv = ratings.groupby("video")[["valence", "arousal", "dominance"]].agg(["mean", "std"])
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2), sharex=True)
for i, dim in enumerate(["valence", "arousal", "dominance"]):
    m, sd = pv[(dim, "mean")], pv[(dim, "std")]
    ax[i].errorbar(pv.index, m, yerr=sd, fmt="o-", capsize=3)
    ax[i].axhline(3, color="crimson", ls="--", lw=1)
    ax[i].set(title=f"{dim} per video (mean ± sd)", xlabel="video", xticks=pv.index)
ax[0].set_ylabel("SAM score")
savefig("06_per_video_ratings.png"); plt.show()

# heatmap subject x video for valence
piv = ratings.pivot(index="subject", columns="video", values="valence")
plt.figure(figsize=(12, 7))
sns.heatmap(piv, cmap="RdYlGn", center=3, annot=True, fmt=".0f", cbar_kws=dict(label="valence"))
plt.title("Valence rating — subject x video"); plt.tight_layout()
savefig("06b_valence_subject_video_heatmap.png"); plt.show()

## 4. Class balance after binarisation

`src/preprocessing.py` labels a trial **high** when `score > 3` (the DREAMER-paper
convention). Below is the balance that classifiers actually see — note how skewed
some dimensions/subjects are.

In [ ]:
THR = 3.0
for dim in ["valence", "arousal", "dominance"]:
    ratings[f"{dim}_bin"] = (ratings[dim] > THR).astype(int)

glob = {d: ratings[f"{d}_bin"].mean() for d in ["valence", "arousal", "dominance"]}
print("Global P(high | score > 3):")
for d, p in glob.items():
    print(f"  {d:10s}: high={p:5.1%}   low={1-p:5.1%}   (n_high={ratings[f'{d}_bin'].sum()}/414)")

fig = plt.figure(figsize=(15, 4.6))
gs = fig.add_gridspec(1, 2, width_ratios=[1, 2.2])
ax0, ax1 = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])

pdf = pd.DataFrame({d: [1 - glob[d], glob[d]] for d in glob}, index=["low (≤3)", "high (>3)"]).T
pdf.plot(kind="barh", stacked=True, ax=ax0, color=["#CC6677", "#117733"])
ax0.set(title="Global class balance (threshold > 3)", xlabel="fraction")
for i, d in enumerate(pdf.index):
    ax0.text(glob[d] / 2 + (1 - glob[d]), i, f"{glob[d]:.0%}", va="center", ha="center", color="w")
ax0.legend(loc="lower left", fontsize=8)

sub_bal = ratings.groupby("subject")[[f"{d}_bin" for d in glob]].mean()
sub_bal.columns = list(glob)
sns.heatmap(sub_bal.T, cmap="RdBu_r", center=.5, vmin=0, vmax=1, annot=True, fmt=".2f",
            annot_kws=dict(fontsize=7), linewidths=.5, ax=ax1, cbar_kws=dict(label="P(high)"))
ax1.set(title=f"P(high) per subject  —  ranges {sub_bal.min().min():.2f} to {sub_bal.max().max():.2f}",
        xlabel="subject")
plt.tight_layout()
savefig("07_class_balance.png"); plt.show()

# count subjects with a degenerate (single-class) target
degen = {d: int(((sub_bal[d] == 0) | (sub_bal[d] == 1)).sum()) for d in glob}
print("\nSubjects with only ONE class for a dimension (subject-dependent CV impossible):", degen)

## 5. EEG signals

### 5.1 Trial durations

Baselines are a fixed 61 s. Stimulus segments vary because the film clips differ in
length (the dataset keeps roughly the last minute of each).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.4))
ax[0].hist(ratings.stim_sec, bins=25, edgecolor="white")
ax[0].set(title="Stimulus segment length", xlabel="seconds", ylabel="# trials")
ax[0].axvline(ratings.stim_sec.mean(), color="crimson", ls="--",
              label=f"mean {ratings.stim_sec.mean():.0f}s")
ax[0].legend()

vid_len = ratings.groupby("video").stim_sec.first()
ax[1].bar(vid_len.index, vid_len.values)
ax[1].set(title="Stimulus length per video (identical across subjects)",
          xlabel="video", ylabel="seconds", xticks=vid_len.index)
savefig("08_trial_durations.png"); plt.show()
print("baseline length (all trials): {:.1f}s constant".format(ratings.base_sec.iloc[0]))
print("stimulus: min {:.0f}s  max {:.0f}s  total {:.0f} min".format(
    ratings.stim_sec.min(), ratings.stim_sec.max(), ratings.stim_sec.sum() / 60))

### 5.2 Raw EEG — what the signal looks like

In [ ]:
def get_eeg(subject, video, kind="stimuli"):
    s = data[subject - 1]
    arr = np.atleast_1d(getattr(s.EEG, kind))[video - 1]
    return np.asarray(arr, float)            # [samples, 14]

# pick, for subject 1, the highest- and lowest-valence videos
s1 = ratings[ratings.subject == 1]
hi_v = int(s1.loc[s1.valence.idxmax(), "video"])
lo_v = int(s1.loc[s1.valence.idxmin(), "video"])

fig, axes = plt.subplots(1, 2, figsize=(15, 7), sharey=True)
for ax, vid, tag in [(axes[0], lo_v, "low"), (axes[1], hi_v, "high")]:
    seg = get_eeg(1, vid, "stimuli")[EEG_FS * 2: EEG_FS * 7]     # a 5-second window
    seg = seg - seg.mean(axis=0, keepdims=True)                  # remove each channel's DC (~4200)
    t = np.arange(seg.shape[0]) / EEG_FS
    spacing = 8 * np.median(seg.std(axis=0))                     # offset scaled to the data
    off = np.arange(14) * spacing
    ax.plot(t, seg + off, lw=.6)
    ax.set_yticks(off); ax.set_yticklabels(CH)
    ax.set_ylim(-spacing, off[-1] + spacing)
    ax.set(title=f"Subject 1 · video {vid} · {tag}-valence (score "
                 f"{int(s1.loc[s1.video==vid,'valence'].iloc[0])})", xlabel="time (s)")
axes[0].set_ylabel("channel (DC removed, offset for display)")
plt.suptitle("Raw EEG stimulus segment — 5 s, 14 channels", y=1.02, fontsize=13)
savefig("09_raw_eeg_traces.png"); plt.show()

### 5.3 Baseline vs stimulus amplitude

The pipeline subtracts the per-channel baseline mean. Here we compare the raw
per-channel **standard deviation** (signal energy) between the neutral baseline and
the emotional stimulus, averaged over trials, for a few subjects.

In [ ]:
def chan_std(subject, kind):
    s = data[subject - 1]
    trials = np.atleast_1d(getattr(s.EEG, kind))
    return np.vstack([np.asarray(tr, float).std(axis=0) for tr in trials]).mean(axis=0)

demo_subs = [1, 5, 10, 15, 20]
base_std = np.vstack([chan_std(s, "baseline") for s in demo_subs])
stim_std = np.vstack([chan_std(s, "stimuli")  for s in demo_subs])

x = np.arange(14)
fig, ax = plt.subplots(figsize=(14, 4.5))
ax.bar(x - .2, base_std.mean(0), .4, yerr=base_std.std(0), capsize=2, label="baseline")
ax.bar(x + .2, stim_std.mean(0), .4, yerr=stim_std.std(0), capsize=2, label="stimulus")
ax.set(xticks=x, xticklabels=CH, ylabel="per-channel std (raw units)",
       title=f"Baseline vs stimulus signal amplitude (subjects {demo_subs})")
ax.legend()
savefig("10_baseline_vs_stimulus.png"); plt.show()

### 5.4 Per-channel signal statistics across all subjects

A quick screen for dead / railed / unusually noisy channels. Values are the raw
stimulus standard deviation per (subject, channel), averaged over that subject's 18
trials.

In [ ]:
all_std = np.vstack([chan_std(s, "stimuli") for s in range(1, N_SUB + 1)])   # [23, 14]
dfe = pd.DataFrame(all_std, index=[f"S{i:02d}" for i in range(1, N_SUB + 1)], columns=CH)

fig, ax = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw=dict(width_ratios=[2.4, 1]))
sns.heatmap(dfe, cmap="mako", annot=False, ax=ax[0],
            cbar_kws=dict(label="raw std"))
ax[0].set(title="Per-channel raw std — subject x channel", xlabel="channel")

med = dfe.median(); mad = (dfe - med).abs().median()
z = ((dfe - med) / (1.4826 * mad + 1e-9))
sns.heatmap(z, cmap="coolwarm", center=0, vmin=-6, vmax=6, ax=ax[1],
            cbar_kws=dict(label="robust z"))
ax[1].set(title="Robust z of the same (|z|>6 ≈ suspicious)", xlabel="channel")
savefig("11_channel_stats_heatmap.png"); plt.show()

flagged = [(dfe.index[i], dfe.columns[j], f"{z.values[i,j]:.1f}")
           for i, j in zip(*np.where(np.abs(z.values) > 6))]
print("cells with |robust z| > 6:", flagged if flagged else "none")

## 6. Frequency-domain view

### 6.1 Average power spectral density (Welch)

Computed on baseline-corrected, 0.3–50 Hz band-passed stimulus signal (same front
end as `preprocessing.py`), pooled over all subjects/trials, per channel.

In [ ]:
def bp_sos(lo, hi, fs=EEG_FS, order=4):
    ny = .5 * fs
    return butter(order, [max(lo/ny, 1e-6), min(hi/ny, .999999)], btype="bandpass", output="sos")

SOS = bp_sos(0.3, 50.0)
NPERSEG = 512
psd_acc = np.zeros((14, NPERSEG // 2 + 1))
n_seg = 0
for si in range(1, N_SUB + 1):
    s = data[si - 1]
    st = np.atleast_1d(s.EEG.stimuli); bl = np.atleast_1d(s.EEG.baseline)
    for ti in range(N_TRIAL):
        x = np.asarray(st[ti], float) - np.asarray(bl[ti], float).mean(0, keepdims=True)
        x = sosfiltfilt(SOS, x, axis=0)
        f, p = welch(x, fs=EEG_FS, nperseg=NPERSEG, axis=0)
        psd_acc += p.T
        n_seg += 1
psd = psd_acc / n_seg
print("averaged over", n_seg, "trials")

bands = {"delta": (1, 4), "theta": (4, 8), "alpha": (8, 14), "beta": (14, 31), "gamma": (31, 50)}
band_col = dict(zip(bands, ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#CCB974"]))
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
for c in range(14):
    ax[0].semilogy(f, psd[c], lw=.8, label=CH[c])
for name, (lo, hi) in bands.items():
    ax[0].axvspan(lo, hi, alpha=.10, color=band_col[name])
    ax[0].text((lo + hi) / 2, 4e3, name, ha="center", fontsize=8)
ax[0].set(xlim=(0, 50), ylim=(1e-1, 2e4), xlabel="Hz", ylabel="PSD (log)",
          title="Mean PSD per channel")
ax[0].legend(ncol=2, fontsize=7, loc="lower left")

mean_psd = psd.mean(0)
ax[1].semilogy(f, mean_psd, "k", lw=1.5)
for name, (lo, hi) in bands.items():
    ax[1].axvspan(lo, hi, alpha=.10, color=band_col[name])
ax[1].axvline(27.9, color="crimson", ls=":", lw=1.2)
ax[1].text(27.9, mean_psd.max(), "  ~28 Hz device artifact", color="crimson", fontsize=8, va="top")
ax[1].set(xlim=(0, 50), ylim=(1e-1, 2e4), xlabel="Hz", ylabel="PSD (log)",
          title="Grand-mean PSD  —  clean 1/f roll-off + narrow ~28 Hz Emotiv artifact")
savefig("12_psd_welch.png"); plt.show()
top = f[np.argsort(mean_psd)[::-1][:8]]
print("strongest spectral components (Hz):", np.round(np.sort(top), 1),
      "\n(the ~28 Hz peak is present in the *raw* signal too — a known Emotiv EPOC artifact; a notch would remove it)")

### 6.2 Band power per channel — scalp topography

In [ ]:
_trapz = getattr(np, "trapezoid", getattr(np, "trapz", None))   # np.trapz removed in NumPy >= 2.1

def band_power(psd, f, lo, hi):
    m = (f >= lo) & (f < hi)
    return _trapz(psd[:, m], f[m], axis=1)            # [14]

bp = {name: band_power(psd, f, lo, hi) for name, (lo, hi) in bands.items()}

# crude 2-D EPOC layout for a topo-style scatter (x=right, y=front), scaled inside the head
_C = {"AF3": (-.25, .90), "AF4": (.25, .90), "F7": (-.80, .55), "F3": (-.40, .55),
      "F4": (.40, .55), "F8": (.80, .55), "FC5": (-.65, .25), "FC6": (.65, .25),
      "T7": (-.95, 0.), "T8": (.95, 0.), "P7": (-.75, -.55), "P8": (.75, -.55),
      "O1": (-.30, -.90), "O2": (.30, -.90)}
xy = np.array([_C[c] for c in CH]) * 0.82        # keep all sensors inside the head circle

fig, axes = plt.subplots(1, 5, figsize=(21, 5.2))
for ax, (name, vals) in zip(axes, bp.items()):
    ax.add_patch(plt.Circle((0, 0), 1.0, fill=False, lw=1.5, zorder=2))
    ax.plot([-.13, 0, .13], [.99, 1.14, .99], "k", lw=1.5, zorder=2)          # nose
    sc = ax.scatter(xy[:, 0], xy[:, 1], c=vals, s=360, cmap="RdBu_r", edgecolor="k", zorder=3)
    for (px, py), c in zip(xy, CH):
        ax.text(px, py, c, ha="center", va="center", fontsize=5.5, zorder=4)
    ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(name); ax.axis("off")
    fig.colorbar(sc, ax=ax, orientation="horizontal", fraction=.05, pad=.04)
plt.suptitle("Grand-mean band power by electrode  (each panel has its own scale)", y=1.0, fontsize=13)
savefig("13_bandpower_topomap.png"); plt.show()

pd.DataFrame(bp, index=CH).round(1)

## 7. Feature-space preview — Differential Entropy

The GCN-family models consume a **14 channel × 5 band DE matrix** per 4-s window
(`differential_entropy` in `preprocessing.py`). One example window below, plus a
classic hypothesis check: **frontal alpha asymmetry** vs valence.

In [ ]:
def differential_entropy(win, fs=EEG_FS):
    de = np.zeros((win.shape[0], len(bands)))
    for b, (lo, hi) in enumerate(bands.values()):
        xb = sosfiltfilt(bp_sos(lo, hi, fs), win, axis=-1)
        de[:, b] = 0.5 * np.log(2 * np.pi * np.e * np.clip(xb.var(-1), 1e-10, None))
    return de                                            # [14, 5]

sig = get_eeg(1, hi_v, "stimuli") - get_eeg(1, hi_v, "baseline").mean(0, keepdims=True)
sig = sosfiltfilt(SOS, sig, axis=0)
w = sig[EEG_FS * 4: EEG_FS * 8].T                        # [14, 512]  one 4-s window
de = differential_entropy(w)

plt.figure(figsize=(6.5, 6))
sns.heatmap(de, xticklabels=list(bands), yticklabels=CH, cmap="viridis", annot=True, fmt=".1f",
            cbar_kws=dict(label="DE"))
plt.title(f"DE matrix — one 4-s window (subj 1, video {hi_v})")
plt.tight_layout(); savefig("14_de_feature_example.png"); plt.show()

In [ ]:
# Frontal Alpha Asymmetry: ln(alpha power F4) - ln(alpha power F3), per trial, vs valence
def faa_for(subject, video):
    x = get_eeg(subject, video, "stimuli") - get_eeg(subject, video, "baseline").mean(0, keepdims=True)
    x = sosfiltfilt(bp_sos(8, 14), x, axis=0)
    pw = x.var(axis=0)
    return np.log(pw[CH.index("F4")]) - np.log(pw[CH.index("F3")])

faa = np.array([faa_for(r.subject, r.video) for r in ratings.itertuples()])
ratings["faa"] = faa
r_faa = np.corrcoef(ratings.faa, ratings.valence)[0, 1]

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
sns.regplot(data=ratings, x="faa", y="valence", ax=ax[0], x_jitter=0, y_jitter=.15,
            scatter_kws=dict(alpha=.35, s=18), line_kws=dict(color="crimson"))
ax[0].set(title=f"Frontal alpha asymmetry (F4−F3) vs valence   r = {r_faa:.2f}",
          xlabel="ln α(F4) − ln α(F3)")
sns.boxplot(data=ratings, x="valence_bin", y="faa", ax=ax[1])
ax[1].set(title="FAA by binarised valence", xticklabels=["low (≤3)", "high (>3)"])
savefig("15_frontal_alpha_asymmetry.png"); plt.show()
print(f"Pearson r(FAA, valence) = {r_faa:.3f}  — weak, as expected for single-feature EEG emotion")

## 8. Data-quality checks

In [ ]:
def _max_true_run(eq):
    if not eq.any():
        return 0
    b = np.concatenate(([0], eq.astype(np.int8), [0]))
    edges = np.flatnonzero(np.diff(b))
    return int((edges[1::2] - edges[::2]).max())

issues = []
nan_trials = inf_trials = flat_channels = big_swing = sat_runs = 0
peak_uv = []
for si in range(1, N_SUB + 1):
    s = data[si - 1]
    for kind in ("baseline", "stimuli"):
        for ti, tr in enumerate(np.atleast_1d(getattr(s.EEG, kind)), start=1):
            a = np.asarray(tr, float)
            if np.isnan(a).any(): nan_trials += 1; issues.append((si, kind, ti, "NaN"))
            if np.isinf(a).any(): inf_trials += 1; issues.append((si, kind, ti, "Inf"))
            ac = a - a.mean(axis=0, keepdims=True)          # remove the ~4200-unit DC first
            if (ac.std(axis=0) < 1e-6).any():
                flat_channels += int((ac.std(axis=0) < 1e-6).sum()); issues.append((si, kind, ti, "flat ch"))
            mx = np.abs(ac).max(); peak_uv.append(mx)
            if mx > 3000: big_swing += 1; issues.append((si, kind, ti, f"|x-DC|max={mx:.0f}"))
            # longest run of identical consecutive samples in any channel (sensor stuck / dropout)
            same = np.diff(a, axis=0) == 0
            longest = max(_max_true_run(same[:, c]) for c in range(a.shape[1]))
            if longest >= 40:
                sat_runs += 1; issues.append((si, kind, ti, f"flat-run={longest}"))

print("(amplitudes are raw Emotiv EPOC units, ~0.51 uV/unit)")
print(f"NaN trials                        : {nan_trials}")
print(f"Inf trials                        : {inf_trials}")
print(f"flat channel-trials              : {flat_channels}")
print(f"trials with |x-DC|max > 3000     : {big_swing}   (extreme excursions / motion / blink bursts)")
print(f"trials with a >=40-sample flat run: {sat_runs}   (possible sensor dropout / quantisation)")
print(f"peak |x-DC|: median {np.median(peak_uv):.0f}   p95 {np.percentile(peak_uv,95):.0f}   max {max(peak_uv):.0f}")
if issues:
    idf = pd.DataFrame(issues, columns=["subject", "kind", "trial", "issue"])
    print("\nissue counts:", idf.issue.str.replace(r"=.*", "", regex=True).value_counts().to_dict())
    display(idf.head(15))

In [ ]:
# outlier trials by total variance (per subject z-score of trial log-variance)
recs = []
for si in range(1, N_SUB + 1):
    s = data[si - 1]
    lv = np.array([np.log(np.asarray(tr, float).var()) for tr in np.atleast_1d(s.EEG.stimuli)])
    zz = (lv - lv.mean()) / (lv.std() + 1e-9)
    for ti, (l, z) in enumerate(zip(lv, zz), start=1):
        recs.append(dict(subject=si, video=ti, logvar=l, z=z))
qc = pd.DataFrame(recs)

plt.figure(figsize=(14, 4))
sns.stripplot(data=qc, x="subject", y="z", size=4)
plt.axhline(3, color="crimson", ls="--"); plt.axhline(-3, color="crimson", ls="--")
plt.title("Per-subject z-score of trial log-variance (|z|>3 = outlier trial)")
plt.tight_layout(); savefig("16_variance_outliers.png"); plt.show()
print("outlier trials (|z|>3):")
print(qc[qc.z.abs() > 3].round(2).to_string(index=False) or "  none")

## 9. Summary — what this means for modelling

| Observation (this EDA) | Implication |
|---|---|
| **23 subjects × 18 trials = 414 ratings only.** 4-s non-overlapping windowing inflates this to ~20 k windows, but windows from one clip are near-duplicates | **split by trial, never by window** — `preprocessing.py` already does (`split_by="trial"`). Treat the effective sample size as ~400 |
| **SAM ratings pile up at 3 and the per-subject range varies a lot** (§3, §3.3) | `score > 3` gives global high-rates of **V 39 % / A 44 % / D 48 %** and far more skew per subject. Report **balanced accuracy / macro-F1**, stratify folds |
| **Arousal ↔ Dominance strongly correlated (r ≈ 0.69); Valence almost independent of both** (§3.2) | arousal and dominance models will look alike; **valence is the genuinely separate target** |
| **Every subject has both classes for all three dims at thr > 3** (§4) | subject-dependent CV is feasible for all 23 — but some subjects are 1:17 imbalanced, so per-subject metrics are noisy |
| **Strong per-subject rater bias and per-channel amplitude differences** (§3.3, §5.3–5.4) | per-subject normalisation is justified; expect a real **LOSO generalisation gap** (the subject of notebook 05) |
| **Baseline fixed 61 s; stimulus 67–394 s** (§5.1) | fixed windows are fine; don't assume equal window counts per trial |
| **Clean 1/f spectrum + physiological bands, plus a narrow ~28 Hz peak already in the raw signal** (§6) | 0.3–50 Hz band-pass is appropriate; consider adding a **notch near 28 Hz** (known Emotiv EPOC artifact) |
| **A handful of high-variance outlier trials and a few high robust-z channel-trials** (§5.4, §8); no NaN/Inf | screen `S20` (frontal channels) and the |z|>3 trials in §8 before trusting their features |
| **Frontal alpha asymmetry alone ≈ 0 correlation with valence (r ≈ −0.09)** (§7) | single hand-picked EEG features are not enough → motivates the multi-channel / multi-band deep models in this repo |

All figures written to `results/figures/eda/` (also embedded above).